In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os


In [2]:
embedings_cache_path = "./outputs/output_embeddings/embeddings_cache_keyed.pkl"
#load from pickle
import pickle
with open(embedings_cache_path, "rb") as f:
    embeddings_cache = pickle.load(f)

#loaded embeddings cache is a dict of text to embedding
print(f"Loaded embeddings cache with {len(embeddings_cache)} entries")

human_embedings = 0
vlm_embedings = 0
for key in embeddings_cache.keys():
    #key is a tuple of agent video question answer
    if "human" in key[0]:
        human_embedings += 1
        #print(f"Human embedding key: {key}")
    else:
        vlm_embedings += 1
        #print(f"VLM embedding key: {key}")
        
        
print(f"Human embedings: {human_embedings}")
print(f"VLM embedings: {vlm_embedings}")

Loaded embeddings cache with 9600 entries
Human embedings: 5600
VLM embedings: 4000


In [3]:
#load answers text cache
answers_text_cache_path = "./data/r2.csv"
df_answers = pd.read_csv(answers_text_cache_path)
#remove duplicates
df_answers = df_answers.drop_duplicates(subset=["AGENT", "VIDEO", "QUESTION_NUM"])
#divide by blocks
def get_block(question_num):
    if question_num <= 5:
        return 1
    elif question_num <= 10:
        return 2
    elif question_num <= 15:
        return 3
    elif question_num <= 20:
        return 4
df_answers["BLOCK"] = df_answers["QUESTION_NUM"].apply(get_block)
print(f"Loaded answers text cache with {len(df_answers)} entries")
print(df_answers.head())

Loaded answers text cache with 9600 entries
          AGENT         VIDEO  QUESTION_NUM QUESTION  \
0  human_lima_1  Robusto2_153             1       Q1   
1  human_lima_2  Robusto2_153             1       Q1   
2  human_lima_3  Robusto2_153             1       Q1   
3  human_lima_4  Robusto2_153             1       Q1   
4  human_lima_5  Robusto2_153             1       Q1   

                                              ANSWER  BLOCK  
0  The ego vehicle is accelerating slowly because...      1  
1            The ego vehicle is turning to the right      1  
2  the ego vehicle brakes and steers slightly to ...      1  
3                                   Braking to yield      1  
4  The ego vehicle is moving forward while mainta...      1  


In [8]:
#first reduce all embedding cache using pca to 2 dimensions and store then into a df with agent video question answer and embedding
from sklearn.decomposition import PCA
from umap import UMAP
embeddings = []
for key, embedding in embeddings_cache.items():
    embeddings.append(embedding.T)

# --- Crear dataframe base ---
keys = list(embeddings_cache.keys())

idx = pd.MultiIndex.from_tuples(keys, names=["AGENT", "VIDEO", "QUESTION_NUM"])
blocks = df_answers.set_index(["AGENT", "VIDEO", "QUESTION_NUM"])["BLOCK"].reindex(idx).to_numpy()

base = pd.DataFrame({
    "AGENT": [k[0] for k in keys],
    "VIDEO": [k[1] for k in keys],
    "QUESTION_NUM": [k[2] for k in keys],
    "BLOCK": blocks,
})

embeddings_arr = np.vstack(embeddings)
coords_PCA = np.zeros((len(keys), 2))
coords_UMAP = np.zeros((len(keys), 2))
for block in [1, 2, 3, 4]:
    mask = base["BLOCK"] == block
    print(f"Processing block {block} with {mask.sum()} entries")
    if mask.any():
        pca_block = PCA(n_components=2)
        umap_block = UMAP(n_components=2)
        coords_PCA[mask.values] = pca_block.fit_transform(embeddings_arr[mask.values])

interactive = base.assign(X=coords_PCA[:, 0], Y=coords_PCA[:, 1])

# --- Merge vectorizado ---
interactive = interactive.merge(
    df_answers[["AGENT", "VIDEO", "QUESTION_NUM", "ANSWER"]],
    on=["AGENT", "VIDEO", "QUESTION_NUM"],
    how="left"
)

def get_group_color(agent):
    if "human" in agent:
        if "nyc" in agent:
            return "nyc"
        else:
            return "lima"
    else:
        return "vlm"
    
color_map = {
    "nyc": "#0000ff",
    "lima": "#ff0000",
    "vlm": "#00ff00",
}
    
    

interactive["group"] = interactive["AGENT"].map(get_group_color)
print(interactive.head())


Processing block 1 with 2400 entries
Processing block 2 with 2400 entries
Processing block 3 with 2400 entries
Processing block 4 with 2400 entries
          AGENT         VIDEO  QUESTION_NUM  BLOCK         X         Y  \
0  human_lima_1  Robusto2_153             1      1  0.165016 -0.223860   
1  human_lima_2  Robusto2_153             1      1  0.160391 -0.173829   
2  human_lima_3  Robusto2_153             1      1  0.135223 -0.213046   
3  human_lima_4  Robusto2_153             1      1  0.130609 -0.073671   
4  human_lima_5  Robusto2_153             1      1  0.222735 -0.113685   

                                              ANSWER group  
0  The ego vehicle is accelerating slowly because...  lima  
1            The ego vehicle is turning to the right  lima  
2  the ego vehicle brakes and steers slightly to ...  lima  
3                                   Braking to yield  lima  
4  The ego vehicle is moving forward while mainta...  lima  


In [9]:
import jscatter
import pandas as pd
import numpy as np

BLOCK_TO_PLOT = int(input("Enter block number to plot (1-4): "))

# Sample data
# Create an interactive scatter plot
scatter = jscatter.Scatter(
    data=interactive[interactive["BLOCK"] == BLOCK_TO_PLOT],  # Filter for block 1
    x='X',
    y='Y',
    color_by='group',
    color_map=color_map,
    opacity=0.7,
    width=800,
    height=600,   
)
scatter.axes(grid=True)
scatter.axes(labels=['PCA 1', 'PCA 2'])
scatter.tooltip(
  enable=True,
  size="medium",
  properties=["AGENT", "VIDEO", "QUESTION_NUM", "ANSWER"],
)

scatter.show()